# Regression, fixed effects, and credibility

[Run in browser](https://muzammilafroz.github.io/applied-economics-data-learning-lab/lab/index.html?path=lessons/04_regression_fixed_effects.ipynb) | [Open in Colab](https://colab.research.google.com/github/muzammilafroz/applied-economics-data-learning-lab/blob/main/notebooks/lessons/04_regression_fixed_effects.ipynb) | [Course home](https://muzammilafroz.github.io/applied-economics-data-learning-lab/) | [Take the test](https://muzammilafroz.github.io/applied-economics-data-learning-lab/tests/?module=module-04)

> This independent learning resource uses fictional, synthetic data. It is not an official assessment or credential.

## Why this lesson matters

Regression summarizes conditional associations, but formula design and sample construction determine what was actually estimated. This lesson makes those choices visible.

Prerequisites: Lessons 1 and 3. You will use Patsy formula syntax and statsmodels, inspect rank redundancy, fit one common sample, use HC1 standard errors, and describe results without causal overreach.

In [1]:
from __future__ import annotations

import os
import sys
import types
from urllib.request import urlopen

if sys.platform == "emscripten":
    import piplite
    await piplite.install("pyodide-http")
    import pyodide_http
    pyodide_http.patch_all()

RAW_CODE_ROOT = "https://raw.githubusercontent.com/muzammilafroz/applied-economics-data-learning-lab/v1.0.1/learning_lab"
if sys.platform == "emscripten":
    from js import window
    if window.location.hostname in {"127.0.0.1", "localhost"}:
        RAW_CODE_ROOT = f"{window.location.origin}/learning_lab"
        os.environ["LEARNING_LAB_DATA_BASE"] = f"{window.location.origin}/data/teaching"

def load_public_module(module_name):
    """Import locally, or fetch the small public helper when running in Colab/Lite."""
    try:
        return __import__(f"learning_lab.{module_name}", fromlist=[module_name])
    except ModuleNotFoundError:
        location = f"{RAW_CODE_ROOT}/{module_name}.py"
        source = urlopen(location).read().decode("utf-8")
        module = types.ModuleType(f"learning_lab.{module_name}")
        exec(compile(source, location, "exec"), module.__dict__)
        return module

lab_io = load_public_module("io")
get_data_url = lab_io.get_data_url
read_teaching_csv = lab_io.read_teaching_csv
read_teaching_geojson = lab_io.read_teaching_geojson

print("Runtime:", sys.platform)
print("Data reference:", os.getenv("LEARNING_LAB_DATA_REF", "v1.0.1"))

Runtime: win32
Data reference: v1.0.1


In [2]:
import numpy as np
import pandas as pd
import patsy
import statsmodels.formula.api as smf

analysis = read_teaching_csv("synthetic_analysis_clean.csv", dtype={"hhid": "string"})
print("Rows before model-specific deletion:", len(analysis))
print(analysis["survey_wave"].value_counts().sort_index())

Rows before model-specific deletion: 1268
survey_wave
Lumen 2014    320
Lumen 2018    308
Noria 2015    300
Noria 2021    340
Name: count, dtype: int64


## Formula syntax

In `outcome ~ predictor + control`, the left side is the outcome and the right side lists explanatory variables. `C(name)` tells Patsy to encode a categorical variable as indicator columns. One level is the reference category when an intercept is present.

In [3]:
formula = "positive_rate ~ wealth_index + C(survey_wave) + head_age + female_head"
model_frame = analysis[[
    "positive_rate", "wealth_index", "survey_wave", "head_age", "female_head"
]].dropna()

print("Common sample rows:", len(model_frame))
print("Rows removed for a missing formula value:", len(analysis) - len(model_frame))

Common sample rows: 1244
Rows removed for a missing formula value: 24


## Why country and year effects are redundant here

The four waves occupy four specific country-year cells. Not every country appears in every year. With this pattern, a full set of survey-wave indicators already represents the observed cells. Adding full country and categorical-year sets creates exact linear dependence among columns.

Matrix rank is the number of linearly independent columns. If rank is below the number of columns, some coefficients cannot be separately identified.

In [4]:
redundant = patsy.dmatrix(
    "C(survey_wave) + C(country) + C(year)",
    analysis,
    return_type="dataframe",
)
wave_only = patsy.dmatrix("C(survey_wave)", analysis, return_type="dataframe")

print("Redundant design: columns =", redundant.shape[1], "rank =", np.linalg.matrix_rank(redundant))
print("Wave design: columns =", wave_only.shape[1], "rank =", np.linalg.matrix_rank(wave_only))
assert np.linalg.matrix_rank(redundant) < redundant.shape[1]
assert np.linalg.matrix_rank(wave_only) == wave_only.shape[1]

Redundant design: columns = 8 rank = 4
Wave design: columns = 4 rank = 4


## Fit OLS with HC1 covariance

Ordinary least squares chooses coefficients that minimize squared residuals. `cov_type="HC1"` replaces the usual homoskedastic covariance estimate with a heteroskedasticity-consistent estimate and a finite-sample adjustment.

HC1 is useful here because the teaching files do not provide survey weights, primary sampling units, or strata. It is not a substitute for those unavailable design fields.

In [5]:
adjusted = smf.ols(formula, data=model_frame).fit(cov_type="HC1")

coefficient = adjusted.params["wealth_index"]
standard_error = adjusted.bse["wealth_index"]
confidence_interval = adjusted.conf_int().loc["wealth_index"].tolist()

print("wealth coefficient:", round(coefficient, 6))
print("HC1 standard error:", round(standard_error, 6))
print("95% confidence interval:", [round(value, 6) for value in confidence_interval])
print("nobs:", int(adjusted.nobs))

wealth coefficient: -0.155773
HC1 standard error: 0.00979
95% confidence interval: [-0.174962, -0.136585]
nobs: 1244


## Interpret scale and language

The outcome is a fraction from zero to one. `wealth_index` was scaled by 100,000, so a one-unit change is meaningful on the synthetic index scale. The coefficient is negative: conditional on the listed controls and wave indicators, a one-unit higher wealth index is associated with a lower positive-rate fraction.

Use `associated with`, not `caused`. Confounding, measurement, selection, and missing survey-design fields remain possible.

In [6]:
coefficient_table = pd.DataFrame(
    {
        "term": adjusted.params.index,
        "estimate": adjusted.params.values,
        "HC1_standard_error": adjusted.bse.values,
        "p_value": adjusted.pvalues.values,
    }
)
display(coefficient_table)

,term,estimate,HC1_standard_error,p_value
0,Intercept,0.324151,0.036645,9.094468e-19
1,C(survey_wave)[T.Lumen 2018],-0.032284,0.026764,2.277322e-01
2,C(survey_wave)[T.Noria 2015],0.059335,0.027652,3.188917e-02
3,C(survey_wave)[T.Noria 2021],0.077319,0.026738,3.831687e-03
4,wealth_index,-0.155773,0.009790,5.299224e-57
5,head_age,-0.000130,0.000652,8.421718e-01
6,female_head,0.003217,0.028994,9.116425e-01


## Report all dummy levels

The coefficient table includes each estimable survey-wave indicator relative to the omitted reference wave. Hiding dummy rows makes the fitted specification harder to audit. A clear report names the reference category and explains that these indicators absorb average differences among waves.

In [7]:
reference_wave = sorted(model_frame["survey_wave"].unique())[0]
dummy_rows = coefficient_table.loc[coefficient_table["term"].str.startswith("C(survey_wave)")]
print("Reference wave:", reference_wave)
display(dummy_rows)

Reference wave: Lumen 2014


,term,estimate,HC1_standard_error,p_value
1,C(survey_wave)[T.Lumen 2018],-0.032284,0.026764,0.227732
2,C(survey_wave)[T.Noria 2015],0.059335,0.027652,0.031889
3,C(survey_wave)[T.Noria 2021],0.077319,0.026738,0.003832


## Sensitivity checks

A sensitivity changes one defensible choice and asks whether the main conclusion is fragile. Here we compare an unadjusted model and a model with region indicators. All reported models use association language.

In [8]:
unadjusted = smf.ols("positive_rate ~ wealth_index", data=analysis).fit(cov_type="HC1")
region_adjusted = smf.ols(
    "positive_rate ~ wealth_index + C(survey_wave) + C(region) + head_age + female_head",
    data=analysis,
).fit(cov_type="HC1")

comparison = pd.DataFrame(
    {
        "model": ["unadjusted", "wave and head controls", "plus region"],
        "wealth_coefficient": [
            unadjusted.params["wealth_index"],
            adjusted.params["wealth_index"],
            region_adjusted.params["wealth_index"],
        ],
        "nobs": [int(unadjusted.nobs), int(adjusted.nobs), int(region_adjusted.nobs)],
    }
)
display(comparison)

,model,wealth_coefficient,nobs
0,unadjusted,-0.150243,1268
1,wave and head controls,-0.155773,1244
2,plus region,-0.156286,1244


## Common failure: mechanical controls

Electricity, floor material, and wall material help construct or closely reflect the simulated wealth index. Controlling for those mechanical components can remove the variation whose association you are trying to describe. Controls should follow an economic argument, not a request to include every available column.

Guided practice: write a two-sentence interview defense. Sentence one should name the estimand and controls. Sentence two should name the strongest limitation and use the word `association`.